# LangChain Tool Rendering Reference

Developer-facing statements defined in `langchain_core.tools.render`.

# `ToolsRenderer`

`ToolsRenderer` represents a callable that converts a list of LangChain tools into one string.

```python
ToolsRenderer = Callable[
    [list[BaseTool]], # Tools supplied to the renderer
    str, # Rendered string returned by the renderer
]
```

A function matching this type can be used wherever LangChain expects a tool-rendering strategy.

---

# `render_text_description`

Renders tool names and descriptions as plain text.

## Signature

```python
render_text_description(
    tools: list[BaseTool], # Tools whose names and descriptions are rendered
) -> str # Return one newline-separated string
```

## Behaviour

- Processes tools in the same order as the supplied list.
- Produces one line for each tool.
- Joins rendered tool lines with newline characters.
- Includes the wrapped Python function signature when the tool exposes a non-empty `func` attribute.
- Omits argument-schema details.
- Returns an empty string when the tool list is empty.

## Output Format

For a tool with an accessible wrapped function:

```text
tool_name(parameter: type) -> return_type - Tool description
```

For a tool without an accessible wrapped function:

```text
tool_name - Tool description
```

## Example Output

```text
search(query: str) -> str - Search for information
calculator(expression: str) -> float - Evaluate a mathematical expression
```

The exact function-signature representation depends on the wrapped callable and its annotations.

---

# `render_text_description_and_args`

Renders tool names, descriptions, and argument schemas as plain text.

## Signature

```python
render_text_description_and_args(
    tools: list[BaseTool], # Tools whose descriptions and argument schemas are rendered
) -> str # Return one newline-separated string
```

## Behaviour

- Processes tools in the same order as the supplied list.
- Produces one line for each tool.
- Joins rendered tool lines with newline characters.
- Includes the wrapped Python function signature when the tool exposes a non-empty `func` attribute.
- Reads each argument schema through `tool.args`.
- Converts the argument-schema dictionary to its normal string representation.
- Returns an empty string when the tool list is empty.

## Output Format

For a tool with an accessible wrapped function:

```text
tool_name(parameter: type) -> return_type - Tool description, args: {argument_schema}
```

For a tool without an accessible wrapped function:

```text
tool_name - Tool description, args: {argument_schema}
```

## Example Output

```text
search(query: str) -> str - Search for information, args: {'query': {'type': 'string'}}
calculator(expression: str) -> float - Evaluate a mathematical expression, args: {'expression': {'type': 'string'}}
```

The rendered argument schema uses Python dictionary formatting rather than JSON formatting.

---

# Comparison

```python
render_text_description(tools) # Render names, callable signatures, and descriptions
render_text_description_and_args(tools) # Also render each tool's argument schema
```

Use `render_text_description` when only a concise tool summary is required.

Use `render_text_description_and_args` when the consumer also needs the names and schemas of accepted arguments.

## Developer-Facing Top-Level Statements

```python
ToolsRenderer # Callable type for tool-rendering functions
render_text_description # Render tool names and descriptions
render_text_description_and_args # Render tool names, descriptions, and argument schemas
```

No public classes, constants, or exceptions are defined in this module.

In [ ]:
from langchain_core.tools import BaseTool, tool # Import the base tool type and tool decorator
from langchain_core.tools.render import ( # Import the public tool-rendering utilities
    ToolsRenderer, # Import the renderer callable type
    render_text_description, # Import the description-only renderer
    render_text_description_and_args, # Import the description-and-schema renderer
) # Finish importing rendering utilities


@tool # Convert the function into a LangChain tool
def add_numbers(first: int, second: int) -> int: # Define an addition tool
    """Add two integer numbers.""" # Provide the tool description
    return first + second # Return the addition result


@tool # Convert the function into a LangChain tool
def greet_user(name: str) -> str: # Define a greeting tool
    """Create a greeting for a user.""" # Provide the tool description
    return f"Hello, {name}!" # Return the greeting message


tools: list[BaseTool] = [add_numbers, greet_user] # Store the tools in rendering order

description_renderer: ToolsRenderer = render_text_description # Store a description-only renderer

schema_renderer: ToolsRenderer = render_text_description_and_args # Store a renderer that includes argument schemas

description_text: str = description_renderer(tools) # Render tool names, signatures, and descriptions

description_and_args_text: str = schema_renderer(tools) # Render descriptions together with argument schemas

empty_result: str = render_text_description([]) # Render an empty tool list

print("Tool descriptions:") # Display the first section heading

print(description_text) # Display the concise tool descriptions

print("\nTool descriptions with arguments:") # Display the second section heading

print(description_and_args_text) # Display descriptions and argument schemas

print("\nEmpty tool list:", repr(empty_result)) # Show that an empty list produces an empty string